In [1]:
from modeling_module.utils.date_util import DateUtil
import polars as pl

MAC_DIR = '/Users/igwanhyeong/PycharmProjects/data_research/raw_data/'


target_raw = (pl.read_parquet(MAC_DIR + 'parquets/dyn_demand.parquet')
                .select(['oper_part_no', 'demand_dt', 'demand_qty'])
                .with_columns(pl.col('demand_dt').cast(pl.Utf8).str.to_date(format='%Y%m%d').alias('demand_dt'))
              )
target_raw_weekly = (target_raw
                    .with_columns(pl.col('demand_dt').map_elements(DateUtil.date_to_yyyyww, return_dtype = pl.Int64).alias('demand_weekly'))
                    .select(['oper_part_no', 'demand_weekly', 'demand_qty'])
                    .group_by(['oper_part_no', 'demand_weekly'])
                    .agg(pl.col('demand_qty').sum().alias('demand_qty'))
)
target_raw_monthly = (target_raw
                      .with_columns(pl.col('demand_dt').map_elements(DateUtil.date_to_yyyymm, return_dtype = pl.Int64).alias('demand_monthly'))
                      .select(['oper_part_no', 'demand_monthly', 'demand_qty'])
                      .group_by(['oper_part_no', 'demand_monthly'])
                      .agg(pl.col('demand_qty').sum().alias('demand_qty'))
                      )


In [2]:
df = target_raw_weekly

all_parts = df['oper_part_no'].unique()
all_periods = pl.Series('demand_weekly', df['demand_weekly'].unique().sort())

# 전체 Cartesian product -> 결측 주차/월 생성
full_df = (
    all_parts.to_frame()
        .join(all_periods.to_frame(), how = 'cross')
)

df_filled = (
    full_df.join(df, on = ['oper_part_no', 'demand_weekly'], how = 'left')
           .fill_null(0.0)
           .with_columns(pl.col('demand_qty').cast(pl.Float64))
           .sort(['oper_part_no', 'demand_weekly'])
)

df_filled

oper_part_no,demand_weekly,demand_qty
str,f64,f64
"""0001-1001""",201752.0,0.0
"""0001-1001""",201801.0,0.0
"""0001-1001""",201802.0,0.0
"""0001-1001""",201803.0,0.0
"""0001-1001""",201804.0,0.0
…,…,…
"""ZZ90239""",202402.0,0.0
"""ZZ90239""",202403.0,0.0
"""ZZ90239""",202404.0,0.0


In [3]:
target_raw_weekly.schema

Schema([('oper_part_no', String),
        ('demand_weekly', Int64),
        ('demand_qty', Float64)])

In [4]:
class DetectSeasonalityUseCase:
    '''
    Detect seasonality for weekly or monthly data with multiple period definitions:
        - S_M  : Monthly seasonality
        - S_Y  : Yearly seasonality
        - S_SP : Spring seasonality
        - S_SM : Summer seasonality
        - S_AU : Autumn seasonality
        - S_WI : Winter seasonality
    '''
    def __init__(
            self,
            target_df: pl.DataFrame,
            period_col: str = 'demand_weekly',
            demand_col: str = 'demand_qty',
            part_col: str = 'oper_part_no',
            periods: list[int] | None = None,
            threshold: float = 0.5,
            min_cycles: int = 2,
    ):


        self.target_df = target_df
        self.part_col = part_col
        self.period_col = period_col # demand_weekly or demand_monthly
        self.demand_col = demand_col
        self.threshold = threshold
        self.min_cycles = min_cycles
        self.periods = periods or [4, 12, 26, 52] # default: monthly, yearly, half-year, weekly-yearly


    # 결측치 주기 보정 (0-fill)
    def _fill_missing_periods(self) -> pl.DataFrame:
        df = self.target_df

        all_parts = df[self.part_col].unique()
        all_periods = pl.Series(self.period_col, df[self.period_col].unique().sort())

        # 전체 Cartesian product -> 결측 주차/월 생성
        full_df = (
            all_parts.to_frame()
                .join(all_periods.to_frame(), how = 'cross')
        )

        df_filled = (
            full_df.join(df, on = [self.part_col, self.period_col], how = 'left')
                   .fill_null(0.0)
                   .with_columns(pl.col(self.demand_col).cast(pl.Float64))
                   .sort([self.part_col, self.period_col])
        )
        return df_filled

    # Seasonality 계산 (Multi period)
    def detect_multi_period_seasonality(self) -> pl.DataFrame:
        try:
            df = self._fill_missing_periods()
            df_sorted = df.sort([self.part_col, self.period_col])

            aggs = [pl.len().alias('n_obs')]
            for p in self.periods:
                aggs.append(
                    pl.corr(
                        pl.col(self.demand_col),
                        pl.col(self.demand_col).shift(p)
                    ).alias(f'corr_lag_{p}')
                )

            stats = df_sorted.group_by(self.part_col, maintain_order = True).agg(aggs)

            # Absolute Transform
            abs_cols = [f'abs_corr_lag_{p}' for p in self.periods]
            stats = stats.with_columns([
                *[pl.col(f"corr_lag_{p}").abs().alias(f"abs_corr_lag_{p}") for p in self.periods]
            ])

            # Find Max correlation
            stats = stats.with_columns(
                pl.max_horizontal([pl.col(c) for c in abs_cols]).alias('best_abs_corr')
            )

            # Decide Best Period
            expr = None
            for p in self.periods:
                cond = pl.col(f'abs_corr_lag_{p}') == pl.col('best_abs_corr')
                if expr is None:
                    expr = pl.when(cond).then(pl.lit(p))
                else:
                    expr = expr.when(cond).then(pl.lit(p))
            expr = expr.otherwise(None)
            stats = stats.with_columns(expr.alias('best_period'))

            # Min cycle and threshold condition
            stats = stats.with_columns(
                (pl.col('n_obs') >= (pl.col('best_period') * self.min_cycles)).alias('enough_length')
            )

            stats = stats.with_columns([
                pl.when((pl.col('best_abs_corr') > self.threshold) & pl.col('enough_length'))
                  .then(pl.lit('Y'))
                  .otherwise(pl.lit('N'))
                  .alias('season_flag')
            ])

            # 계절성 타입 매핑
            period_label_map = {
                4: "S_M",
                12: "S_Y",
                13: "S_SP",
                26: "S_SM",
                39: "S_AU",
                52: "S_WI"
            }

            stats = stats.with_columns([
                pl.when(pl.col('season_flag') == 'Y')
                  .then(
                    pl.col('best_period').map_elements(lambda x: period_label_map.get(x, 'S_Y'))
                  )
                  .otherwise(pl.lit('Non-Seasonal'))
                  .alias('season_type')
            ])
            return stats.select([self.part_col, 'best_period', 'best_abs_corr', 'season_flag', 'season_type'])
        except Exception as e:
            print(e)
            e.traceback()
            return pl.DataFrame(
                schema = [
                    (self.part_col, pl.Utf8),
                    ('best_period', pl.Int64),
                    ('best_abs_corr', pl.Float64),
                    ('season_flag', pl.Utf8),
                    ('season_type', pl.Utf8)
                ]
            )

In [5]:
weekly_uc = DetectSeasonalityUseCase(
    target_df = target_raw_weekly,
    part_col = 'oper_part_no',
    period_col = 'demand_weekly',
    demand_col = 'demand_qty',
    periods = [13, 26, 39, 52],
    threshold = 0.5,
    min_cycles = 2
)
season_weekly = weekly_uc.detect_multi_period_seasonality()

/var/folders/py/xgf_87rd5nz9rsbc143wp9qc0000gn/T/ipykernel_36264/312218748.py:114: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  stats = stats.with_columns([


In [6]:
season_weekly

oper_part_no,best_period,best_abs_corr,season_flag,season_type
str,i32,f64,str,str
"""0001-1001""",39,0.006095,"""N""","""Non-Seasonal"""
"""0001-1002""",13,0.010156,"""N""","""Non-Seasonal"""
"""0001-1005""",39,0.005727,"""N""","""Non-Seasonal"""
"""0001-1006""",52,NaN,"""Y""","""S_WI"""
"""0011-2-1-04""",52,0.022901,"""N""","""Non-Seasonal"""
…,…,…,…,…
"""ZZ90207""",52,0.131343,"""N""","""Non-Seasonal"""
"""ZZ90222R""",52,0.007519,"""N""","""Non-Seasonal"""
"""ZZ90237""",52,0.029727,"""N""","""Non-Seasonal"""


In [7]:
season_weekly.select('season_type').unique()

season_type
str
"""S_SM"""
"""Non-Seasonal"""
"""S_WI"""
"""S_AU"""
"""S_SP"""


In [3]:
import sys

import polars as pl
import torch

from modeling_module.data_loader.MultiPartDataModule import MultiPartDataModule
from modeling_module.models.PatchMixer.common.configs import PatchMixerConfig
from modeling_module.models.PatchTST.common.configs import PatchTSTConfig, PatchTSTConfigWeekly
from modeling_module.utils.checkpoint import save_model_dict, load_model_dict

'''
pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
https://developer.nvidia.com/cuda-12-8-0-download-archive
'''

MAC_DIR = '/Users/igwanhyeong/PycharmProjects/data_research/raw_data/'
WINDOW_DIR = 'C:/Users/USER/PycharmProjects/research/raw_data/'

if sys.platform == 'win32':
    DIR = WINDOW_DIR
    print(torch.cuda.is_available())
    print(torch.cuda.device_count())
    print(torch.version.cuda)
    print(torch.__version__)
    print(torch.cuda.get_device_name(0))
    print(torch.__version__)
else:
    DIR = MAC_DIR

save_dir = DIR + 'fit/20251106_running'


target_dyn_demand_weekly = pl.read_parquet(DIR + 'target_dyn_demand_weekly.parquet').sort(['oper_part_no', 'demand_dt'])
target_dyn_demand_weekly = (target_dyn_demand_weekly.group_by('oper_part_no', maintain_order = True).map_groups(lambda g: g.with_columns(pl.arange(1, len(g) + 1).alias('seq'))))

filtered_target = target_dyn_demand_weekly.group_by('oper_part_no').agg(pl.col('seq').max().alias('seq_max')).filter(pl.col('seq_max') > 260).select('oper_part_no')

target_dyn_demand_weekly = target_dyn_demand_weekly.join(filtered_target, on = 'oper_part_no', how = 'right').select(['oper_part_no', 'demand_dt', 'demand_qty'])
target_dyn_demand_weekly.describe()

True
1
12.8
2.9.0.dev20250716+cu128
NVIDIA GeForce RTX 5080
2.9.0.dev20250716+cu128


statistic,oper_part_no,demand_dt,demand_qty
str,str,f64,f64
"""count""","""320080""",320080.0,320080.0
"""null_count""","""0""",0.0,0.0
"""mean""",null,202232.232279,54.808988
"""std""",null,261.522675,778.060915
"""min""","""01070-51445""",201752.0,0.0
"""25%""",null,202016.0,3.0
"""50%""",null,202226.0,8.0
"""75%""",null,202444.0,24.0
"""max""","""ULPK0041""",202705.0,260700.0
